In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

from dask.diagnostics import ProgressBar
import dask.dataframe as dd
import pandas as pd
import numpy as np
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster
from src.utils.stochastic_models import OrnsteinUhlenbeck
import statsmodels.api as sm
from mc_postgres_db.operations import set_data

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

CLUSTER_TYPE = "local"
N_WORKERS = 6

engine = create_engine(POSTGRES_URL)

In [ ]:
max_groups = 25
lookback_days = 30

In [ ]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=lookback_days)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

In [ ]:
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
# Load market data with pandas (before cluster)
print("Loading market data...")
market_data = pd.read_sql(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    )
    .where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive))
    .order_by(models.ProviderAssetMarket.timestamp),
    engine,
)
print(f"Market data loaded: {len(market_data)} rows")

# Load provider asset group members (filtered)
print("Loading provider asset group members...")
members_data = pd.read_sql(
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    ).where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    ),
    engine,
)
print(f"Members data loaded: {len(members_data)} rows")

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="3GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="prefect-cluster",
        n_workers=N_WORKERS,
        region="us-east-1",
        container="ghcr.io/manning-capital/mc-notebooks:main",
        worker_memory="16GB",
        worker_cpu=2,
    )

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
@delayed
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    members_chunk: pd.DataFrame,
    market_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load the pairs trading frame for a chunk of provider asset groups.
    Returns only the essential columns needed for cointegration analysis.

    Args:
        provider_asset_group_ids: List of provider asset group IDs to process
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        members_chunk: Pre-filtered DataFrame of provider asset group members
        market_data: Broadcasted market data DataFrame

    Returns:
        pandas DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Step 1: Generate timeframe using pd.date_range
    time_frame = pd.DataFrame({
        'timestamp': pd.date_range(start, end, freq='1min')
    })

    # Step 2: Use passed-in members_chunk (already filtered)
    members = members_chunk

    # Step 3: Cross join
    full_frame = time_frame.merge(members, how="cross")
    full_frame = full_frame.sort_values("timestamp")

    # Step 4: Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame,
        market_data,
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Step 5: Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})
    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    members_data: pd.DataFrame,
    market_data_future,
    n_workers: int = 10,
) -> dd.DataFrame:
    """
    Get the pairs trading frame with only essential columns for cointegration analysis.

    Args:
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        provider_asset_group_ids: List of provider asset group IDs to process
        members_data: Pre-loaded DataFrame of provider asset group members
        market_data_future: Broadcasted market data future
        n_workers: Number of parallel workers

    Returns:
        Dask DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Split provider asset groups into chunks
    n_chunks = min(n_workers, len(provider_asset_group_ids))
    group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

    # Create delayed tasks with filtered member chunks
    delayed_dfs = []
    for chunk in group_chunks:
        # Filter members data for this specific chunk
        members_chunk = members_data[
            members_data['provider_asset_group_id'].isin(chunk.tolist())
        ].copy()
        
        delayed_dfs.append(
            load_pairs_trading_frame_chunk(
                start, end, members_chunk, market_data_future
            )
        )

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)

    # Set index to provider_asset_group_id
    pairs_trading_frame = pairs_trading_frame.set_index(
        "provider_asset_group_id", sorted=True
    )

    return pairs_trading_frame

In [ ]:
pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)

In [ ]:
with ProgressBar():
    cointegration_p_values_computed = cointegration_p_values.compute()
cointegration_p_values_computed

In [ ]:
cointegrated_provider_asset_group_ids = cointegration_p_values_computed.loc[
    cointegration_p_values_computed["p_value"] < 0.001
].index.tolist()
print(
    f"Cointegrated provider asset group ids (count: {len(cointegrated_provider_asset_group_ids)}): {cointegrated_provider_asset_group_ids}"
)

In [ ]:
def get_cointegrated_stats(df: pd.DataFrame) -> pd.Series:
    """
    Get the cointegrated stats for a given dataframe.
    """

    # Compute the linear regression.
    X = df["close_1"].to_numpy()
    y = df["close_2"].to_numpy()
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()

    # Get the residuals.
    linear_fit_alpha = results.params[0]
    linear_fit_beta = results.params[1]
    linear_fit_mse = results.mse_total
    linear_fit_r_squared = results.rsquared
    linear_fit_r_squared_adj = results.rsquared_adj
    residuals = results.resid

    # Get the cointegration stats.
    ou_params = OrnsteinUhlenbeck().fit(residuals)

    return pd.Series(
        [
            linear_fit_alpha,
            linear_fit_beta,
            linear_fit_mse,
            linear_fit_r_squared,
            linear_fit_r_squared_adj,
            ou_params.mu,
            ou_params.theta,
            ou_params.sigma,
        ],
        index=[
            "linear_fit_alpha",
            "linear_fit_beta",
            "linear_fit_mse",
            "linear_fit_r_squared",
            "linear_fit_r_squared_adj",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
        ],
        dtype=float,
    )


In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
cointegrated_pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    cointegrated_provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
cointegrated_pairs_trading_stats = cointegrated_pairs_trading_frame.groupby(
    "provider_asset_group_id"
)[["close_1", "close_2"]].apply(
    lambda df: get_cointegrated_stats(df),
    meta={
        "linear_fit_alpha": pd.Series([], dtype=float),
        "linear_fit_beta": pd.Series([], dtype=float),
        "linear_fit_mse": pd.Series([], dtype=float),
        "linear_fit_r_squared": pd.Series([], dtype=float),
        "linear_fit_r_squared_adj": pd.Series([], dtype=float),
        "ou_mu": pd.Series([], dtype=float),
        "ou_theta": pd.Series([], dtype=float),
        "ou_sigma": pd.Series([], dtype=float),
    },
)

In [ ]:
cointegrated_pairs_trading_stats_computed = cointegrated_pairs_trading_stats.compute()
cointegrated_pairs_trading_stats_computed

In [ ]:
# cointegration_p_values_computed.to_csv("cointegration_p_values.csv")
# cointegrated_pairs_trading_stats_computed.to_csv("cointegrated_pairs_trading_stats.csv")
# cointegration_p_values_computed.to_parquet("cointegration_p_values.parquet")
# cointegrated_pairs_trading_stats_computed.to_parquet(
#     "cointegrated_pairs_trading_stats.parquet"
# )


In [ ]:
toset = cointegration_p_values_computed.merge(
    cointegrated_pairs_trading_stats_computed, left_index=True, right_index=True
).reset_index()
toset = toset.rename(columns={"p_value": "cointegration_p_value"})
toset["lookback_window_seconds"] = 30 * 24 * 60 * 60
toset["timestamp"] = end_naive
toset = toset[
    [
        "timestamp",
        "provider_asset_group_id",
        "lookback_window_seconds",
        "cointegration_p_value",
        "linear_fit_alpha",
        "linear_fit_beta",
        "linear_fit_mse",
        "linear_fit_r_squared",
        "linear_fit_r_squared_adj",
        "ou_mu",
        "ou_theta",
        "ou_sigma",
    ]
]
toset


In [ ]:
import matplotlib.pyplot as plt
data = cointegrated_pairs_trading_frame.reset_index().compute().merge(
    toset.drop(columns=["lookback_window_seconds", "timestamp"]), on="provider_asset_group_id"
)
data = data.loc[
    data["provider_asset_group_id"] == cointegrated_provider_asset_group_ids[1]
]
close_1 = data["close_1"].to_numpy()
close_2 = data["close_2"].to_numpy()
residuals = close_2 - data["linear_fit_alpha"] + data["linear_fit_beta"] * close_1
plt.plot(residuals)
plt.show()

In [ ]:
set_data(
    engine,
    models.ProviderAssetGroupAttribute.__tablename__,
    toset,
    operation_type="upsert",
)

In [ ]:
cluster.close(force_shutdown=True)
